# 01 — Historical weather data collection

This notebook downloads daily ERA5 weather data from the Open-Meteo Historical Weather API for the Italian cities selected in the project plan.

The notebook preserves two files for every city:

- a JSON file containing the complete API response;
- a CSV file containing one normalized row per city and day.

The complete period is **1 January 1940 through 31 December 2025**. Existing complete downloads are skipped to avoid unnecessary API requests.

## 1. Project setup

The following cell locates the repository root. This allows the notebook to import the reusable functions in `scripts` whether Jupyter starts from the repository root or from the `notebooks` directory.

In [1]:
from pathlib import Path
import csv
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "scripts").is_dir():
    raise RuntimeError(
        "Could not locate the project root. Start Jupyter from the "
        "Weather-Report repository or its notebooks directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/roberto/Developer/aruzhan-uni-projects/Weather-Report


In [2]:
from scripts import (
    DEFAULT_DAILY_VARIABLES,
    ArchiveRequest,
    OpenMeteoError,
    build_archive_url,
    fetch_city_weather,
    get_cities,
    save_records_csv,
    save_response_json,
)

## 2. Download configuration

ERA5 is used for every city and year to maintain a consistent data source across the full historical period. The end date is the last complete year, so partial 2026 observations cannot distort annual comparisons.

In [3]:
from datetime import date

START_DATE = date(1940, 1, 1)
END_DATE = date(2025, 12, 31)
DAILY_VARIABLES = DEFAULT_DAILY_VARIABLES
CITIES = get_cities()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

def data_file_stem(city):
    """Return the reproducible file stem used for one city download."""
    return (
        f"{city.slug}_{START_DATE:%Y%m%d}_{END_DATE:%Y%m%d}_era5"
    )

print(f"Period: {START_DATE} to {END_DATE}")
print(f"Variables: {', '.join(DAILY_VARIABLES)}")
print(f"Cities: {', '.join(city.name for city in CITIES)}")
print(f"Raw data directory: {RAW_DATA_DIR}")

Period: 1940-01-01 to 2025-12-31
Variables: temperature_2m_mean, temperature_2m_max, temperature_2m_min, precipitation_sum
Cities: Milan, Turin, Rome, Bari, Palermo, Cagliari
Raw data directory: /Users/roberto/Developer/aruzhan-uni-projects/Weather-Report/data/raw


## 3. Inspect the planned requests

This cell only builds and displays request URLs. It does not contact the API. Inspecting the parameters before downloading makes the data acquisition step easier to reproduce and audit.

In [4]:
for city in CITIES:
    request = ArchiveRequest(
        city=city,
        start_date=START_DATE,
        end_date=END_DATE,
        daily_variables=DAILY_VARIABLES,
    )
    print(f"{city.name}: {build_archive_url(request)}\n")

Milan: https://archive-api.open-meteo.com/v1/archive?latitude=45.4642&longitude=9.19&start_date=1940-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Europe%2FRome&models=era5

Turin: https://archive-api.open-meteo.com/v1/archive?latitude=45.0703&longitude=7.6869&start_date=1940-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Europe%2FRome&models=era5

Rome: https://archive-api.open-meteo.com/v1/archive?latitude=41.9028&longitude=12.4964&start_date=1940-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Europe%2FRome&models=era5

Bari: https://archive-api.open-meteo.com/v1/archive?latitude=41.1171&longitude=16.8719&start_date=1940-01-01&end_date=2025-12-31&daily=temperature_2m_mean%2Ctemperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Europe%2FRome&mod

## 4. Download and save the data

Running the following cell makes one API request per city. A temporary API or network failure is retried automatically. If Open-Meteo returns HTTP 429, the cell displays a warning, waits at least one minute and then continues on its own; do not interrupt it during this pause.

If both the JSON and CSV files for a city already exist, that city is skipped. If only one of the two files exists, the data are downloaded again and only the missing file is created. Existing files are never overwritten by this notebook.

In [5]:
downloaded_datasets = {}
skipped_cities = []

for position, city in enumerate(CITIES, start=1):
    stem = data_file_stem(city)
    json_path = RAW_DATA_DIR / f"{stem}.json"
    csv_path = RAW_DATA_DIR / f"{stem}.csv"

    if json_path.exists() and csv_path.exists():
        skipped_cities.append(city.slug)
        print(f"[{position}/{len(CITIES)}] {city.name}: files already exist, skipped.")
        continue

    print(f"[{position}/{len(CITIES)}] Downloading {city.name}...")
    try:
        dataset = fetch_city_weather(
            city,
            start_date=START_DATE,
            end_date=END_DATE,
            daily_variables=DAILY_VARIABLES,
            max_retries=5,
        )
    except OpenMeteoError as error:
        raise RuntimeError(f"Download failed for {city.name}: {error}") from error

    if not json_path.exists():
        save_response_json(dataset, json_path)
    if not csv_path.exists():
        save_records_csv(dataset, csv_path)

    downloaded_datasets[city.slug] = dataset
    print(f"    Saved {len(dataset):,} daily observations.")

print("\nDownload step completed.")
print(f"New downloads: {len(downloaded_datasets)}")
print(f"Skipped existing downloads: {len(skipped_cities)}")

[1/6] Milan: files already exist, skipped.
[2/6] Downloading Turin...
    Saved 31,412 daily observations.
[3/6] Downloading Rome...


/Users/roberto/Developer/aruzhan-uni-projects/Weather-Report/scripts/open_meteo.py:123: RuntimeWarning: Open-Meteo minute limit reached. Retrying automatically in 60 seconds.
  payload = _fetch_json(


    Saved 31,412 daily observations.
[4/6] Downloading Bari...
    Saved 31,412 daily observations.
[5/6] Downloading Palermo...
    Saved 31,412 daily observations.
[6/6] Downloading Cagliari...
    Saved 31,412 daily observations.

Download step completed.
New downloads: 5
Skipped existing downloads: 1


## 5. Validate the saved CSV files

The final check counts daily rows and compares them with the number of calendar days requested. It also reports file sizes and missing files. This is an initial integrity check; detailed quality checks will be performed in the cleaning notebook.

In [6]:
expected_rows = (END_DATE - START_DATE).days + 1
validation_results = []

for city in CITIES:
    stem = data_file_stem(city)
    csv_path = RAW_DATA_DIR / f"{stem}.csv"
    json_path = RAW_DATA_DIR / f"{stem}.json"

    if not csv_path.exists() or not json_path.exists():
        validation_results.append(
            {"city": city.name, "status": "missing file", "rows": None}
        )
        continue

    with csv_path.open(encoding="utf-8", newline="") as file_handle:
        row_count = sum(1 for _ in csv.DictReader(file_handle))

    validation_results.append(
        {
            "city": city.name,
            "status": "ok" if row_count == expected_rows else "unexpected rows",
            "rows": row_count,
            "expected_rows": expected_rows,
            "csv_megabytes": round(csv_path.stat().st_size / 1_000_000, 2),
            "json_megabytes": round(json_path.stat().st_size / 1_000_000, 2),
        }
    )

for result in validation_results:
    print(result)

{'city': 'Milan', 'status': 'ok', 'rows': 31412, 'expected_rows': 31412, 'csv_megabytes': 2.23, 'json_megabytes': 2.08}
{'city': 'Turin', 'status': 'ok', 'rows': 31412, 'expected_rows': 31412, 'csv_megabytes': 2.3, 'json_megabytes': 2.08}
{'city': 'Rome', 'status': 'ok', 'rows': 31412, 'expected_rows': 31412, 'csv_megabytes': 2.24, 'json_megabytes': 2.09}
{'city': 'Bari', 'status': 'ok', 'rows': 31412, 'expected_rows': 31412, 'csv_megabytes': 2.25, 'json_megabytes': 2.09}
{'city': 'Palermo', 'status': 'ok', 'rows': 31412, 'expected_rows': 31412, 'csv_megabytes': 2.47, 'json_megabytes': 2.1}
{'city': 'Cagliari', 'status': 'ok', 'rows': 31412, 'expected_rows': 31412, 'csv_megabytes': 2.47, 'json_megabytes': 2.09}


## Next step

The next notebook will load the saved CSV files, combine all cities into one tidy dataset, verify dates and units, detect missing or duplicate observations, and save a cleaned dataset in `data/processed`.